# Merge two datasets

The common shape of "we got more data": a second export, in the same task with
almost the same labels. Almost is the whole problem — `Positive` and `positive`
are two classes to a model and one to a person, and nobody notices until the
confusion matrix has a row with four items in it.

In [ ]:
import orinth
import polars as pl

candidates = [
    dataset
    for dataset in orinth.datasets.list()
    if dataset.task_type == "text_classification" and dataset.total_items > 0
]
for dataset in candidates:
    print(f"{dataset.id:42} {dataset.total_items:>7} items  {dataset.labels}")

A, B = candidates[0], candidates[1]

In [ ]:
left = orinth.datasets.load(A.id, ["train", "valid", "test"], limit=5000).select(["text", "label"])
right = orinth.datasets.load(B.id, ["train", "valid", "test"], limit=5000).select(["text", "label"])
print(A.name, left.shape, "|", B.name, right.shape)

## Reconcile the labels

Before concatenating, look at what each side calls things. A label in one and
not the other is either a rename to fix or a genuinely new class to keep — and
that is a decision, not something to let `concat` make for you.

In [ ]:
left_labels = set(left["label"].unique().to_list())
right_labels = set(right["label"].unique().to_list())

print("shared    :", sorted(left_labels & right_labels))
print("only in A :", sorted(left_labels - right_labels))
print("only in B :", sorted(right_labels - left_labels))

# Case and whitespace are the usual culprits; state the mapping rather than
# normalising blindly, so a real rename stays visible in the diff.
RENAME = {}
normalise = pl.col("label").str.strip_chars().str.to_lowercase().replace(RENAME)
left = left.with_columns(normalise)
right = right.with_columns(normalise)
print("\nafter:", sorted(set(left['label'].unique().to_list()) | set(right['label'].unique().to_list())))

## Concatenate, then de-duplicate across the seam

The join is where duplicates come from: the same row in both exports, or a row
you already cleaned once. Dropping them after the merge is the only place it can
be done, because neither side has the pair.

In [ ]:
merged = pl.concat([left, right], how="vertical")
before = merged.height
merged = merged.filter(pl.col("text").str.strip_chars().str.len_chars() > 0).unique(subset=["text"])
print(f"{before} -> {merged.height} rows ({before - merged.height} empty or duplicated)")

merged.group_by("label").agg(pl.len().alias("rows")).sort("rows", descending=True)

## Register the union

A **new** dataset, always — `register()` never writes into an existing id.
Rebuilding a dataset under whatever is reading it is the failure this platform
already lived through once, and a notebook is the easiest place to trigger it by
accident.

In [ ]:
ref = orinth.datasets.register(
    merged,
    name=f"{A.name} + {B.name}",
    task_type="text_classification",
    columns={"text": "text", "label": "label"},
    split={"train": 0.7, "valid": 0.2, "test": 0.1},
)
print(ref.id)
print(ref.splits)
print(ref.readiness.state, "|", ref.readiness.summary)